# 02 · Cleaning (Ludovico)

**Goal:** fix every value the EDA showed to be wrong, using **fixed rules that do not learn from the data**.

**Ground rules**
- Same rules for train and test → one `clean(df)` function.
- Wrong value → `NaN` (imputed later inside the Pipeline, `04_preprocessing`), unless an **exact identity** gives the true value back.
- Never drop test rows. Drop train rows only when they are corrupted records.
- Not here: imputation, scaling, encoding, percentile-based outliers, feature selection (they learn from data).

<div class="alert alert-block alert-info">

# TOC<a class="anchor" id="toc"></a>
* [<font color='#E8800A'>Setup</font>](#setup)
* [<font color='#E8800A'>1. Names and types</font>](#names)
* [<font color='#E8800A'>2. Empty rows (train only)</font>](#empty)
* [<font color='#E8800A'>3. Impossible values → NaN</font>](#invalid)
* [<font color='#E8800A'>4. Rebuild values from exact identities</font>](#rebuild)
* [<font color='#E8800A'>5. Categorical labels</font>](#categories)
* [<font color='#E8800A'>6. The full `clean()` function and checks</font>](#clean)
* [<font color='#E8800A'>7. Summary</font>](#summary)

</div>

# <font color='#E8800A'>Setup</font> <a class="anchor" id="setup"></a>
[Back to TOC](#toc)

In [1]:
# Make the project root importable so that `src` can be imported from any notebook folder
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "environment.yml").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Could not find environment.yml in any parent folder.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.paths import DATA_RAW

In [2]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)

TARGET = "repeat_purchase_90d"
PLACEHOLDER = 1e9  # value used in the raw data in place of a real amount
TOL = 0.01         # one cent / a hundredth of a day: tolerance for float comparisons

In [3]:
train_raw = pd.read_csv(DATA_RAW / "train.csv", index_col="ID")
test_raw = pd.read_csv(DATA_RAW / "test.csv", index_col="ID")

print(f"train: {train_raw.shape[0]:,} rows x {train_raw.shape[1]} columns")
print(f"test:  {test_raw.shape[0]:,} rows x {test_raw.shape[1]} columns")

train: 6,534 rows x 38 columns
test:  986 rows x 37 columns


In [4]:
def side_by_side(log_train, log_test, keys):
    """Merge the train and test logs of a step into one table (rows affected in each set)."""
    table = log_train.merge(log_test, on=keys, how="outer", suffixes=(" train", " test")).fillna(0)
    value_cols = [c for c in table.columns if c not in keys]
    table[value_cols] = table[value_cols].astype(int)
    return table[table[value_cols].sum(axis=1) > 0].reset_index(drop=True)

# <font color='#E8800A'>1. Names and types</font> <a class="anchor" id="names"></a>
[Back to TOC](#toc)

**Problem:** 9 columns have Portuguese names with spaces and trailing underscores (`Total Imp_`). They are awkward to type and inconsistent with the rest.

**Fix**
- Rename them to English `snake_case` (mapping table below). The other columns are already fine.
- `purchase_date` → `datetime`.
- Categoricals → `str` (missing values stay `NaN`, which sklearn imputers understand), with leading/trailing spaces stripped.

In [5]:
COLUMN_MAP = {
    "Total a Pagar": "total_to_pay",        # amount paid by the customer = net + tax
    "Total Bruto": "total_gross",           # before discounts
    "Total Liquido": "total_net",           # after discounts, before tax
    "Total Imp_": "total_tax",              # VAT
    "Total Portes": "total_shipping",       # shipping (not part of total_to_pay)
    "Total Desc_ Global": "discount_global",
    "Total Desc_ Linha": "discount_line",
    "Custo": "cost",                        # cost of goods for the company
    "N_ Detalhes": "n_lines",               # number of product lines on the invoice
}
pd.Series(COLUMN_MAP, name="new name").rename_axis("original name").to_frame()

,new name
original name,
Total a Pagar,total_to_pay
Total Bruto,total_gross
Total Liquido,total_net
Total Imp_,total_tax
Total Portes,total_shipping
Total Desc_ Global,discount_global
Total Desc_ Linha,discount_line
Custo,cost
N_ Detalhes,n_lines


In [6]:
CATEGORICAL = ["payment_type", "salesperson", "commercial_zone", "transport_method",
               "warehouse", "document_series"]
WINDOWS = [30, 90, 180, 365]

# Column families used by the rules below (names after renaming)
DAY_COLS = ["days_since_previous_purchase", "customer_tenure_days",
            "prior_mean_gap_days", "prior_std_gap_days"]
COUNT_COLS = (["n_lines", "invoice_count_current_purchase", "prior_purchase_count"]
              + [f"purchase_count_{w}d" for w in WINDOWS])
AMOUNT_COLS = (["total_to_pay", "total_gross", "total_net", "total_tax", "total_shipping",
                "discount_global", "discount_line", "cost", "prior_total_spend",
                "prior_mean_purchase_value", "prior_std_purchase_value",
                "prior_min_purchase_value", "prior_max_purchase_value"]
               + [f"spend_{w}d" for w in WINDOWS])


def rename_and_type(df):
    """English snake_case names, datetime date, stripped string categoricals."""
    df = df.rename(columns=COLUMN_MAP)
    df["purchase_date"] = pd.to_datetime(df["purchase_date"])
    for col in CATEGORICAL:
        df[col] = df[col].astype("str").str.strip()  # pandas default string type: missing = NaN (sklearn-friendly)
    return df

In [7]:
train_1 = rename_and_type(train_raw)
test_1 = rename_and_type(test_raw)

assert set(DAY_COLS + COUNT_COLS + AMOUNT_COLS + CATEGORICAL) <= set(train_1.columns)
train_1.dtypes.value_counts()

float64           29
str                6
int64              2
datetime64[us]     1
Name: count, dtype: int64

# <font color='#E8800A'>2. Empty rows (train only)</font> <a class="anchor" id="empty"></a>
[Back to TOC](#toc)

**Problem (EDA Step 4):** 65 train rows have **every feature missing**. Only `customer_id` and the target are filled in. They are also the source of all 10 duplicated rows.

**Fix:** drop them from **train only**.
- A model cannot learn anything from a row with no features.
- Test has no such rows, and test rows are never dropped anyway.

In [8]:
def drop_empty_rows(df):
    """Remove rows where every feature is missing (train only). Returns the frame and the number dropped."""
    features = df.columns.difference([TARGET, "customer_id"])
    empty = df[features].isna().all(axis=1)
    return df[~empty], int(empty.sum())


train_2, n_empty = drop_empty_rows(train_1)
test_2 = test_1  # nothing dropped from test

print(f"empty rows dropped from train: {n_empty}")
print(f"empty rows in test:            {drop_empty_rows(test_1)[1]}")
print(f"duplicated rows in train, before -> after: {train_1.duplicated().sum()} -> {train_2.duplicated().sum()}")

empty rows dropped from train: 65
empty rows in test:            0
duplicated rows in train, before -> after: 10 -> 0


# <font color='#E8800A'>3. Impossible values → NaN</font> <a class="anchor" id="invalid"></a>
[Back to TOC](#toc)

**Problem:** some values break **domain logic**. They look like errors injected one column at a time, in train **and** test.

| Pattern | Columns | Why it is impossible |
|---|---|---|
| Negative day interval (always exactly **−4**) | `days_since_previous_purchase`, `prior_mean_gap_days` | time cannot be negative |
| Placeholder **1,000,000,000** | `prior_mean_purchase_value`, `spend_365d` | not a real purchase value |
| Negative amount (sign flipped) | `prior_mean_purchase_value`, `spend_365d` | spend cannot be negative |
| Mean ≠ total / count (≈ 8–25× too large) | `prior_mean_purchase_value` | the mean **is** total spend / number of purchases |
| Max > total spend | `prior_max_purchase_value` | one purchase cannot exceed the sum of all purchases |
| 365-day spend > lifetime spend, or < 180-day spend | `spend_365d` | the 365-day window is inside the full history and contains the 180-day window |

The first three were found in the EDA. The last three were found while preparing this notebook, by checking identities between columns.

**Fix:** set the value to `NaN`.
- The rules are written as **general domain rules** ("no negative amounts"), not hard-coded to −4. A new error of the same kind in test would still be caught.
- Section 4 then gives back the values that can be rebuilt exactly.

In [9]:
def invalid_value_rules(df):
    """List of (rule, column, mask). True in the mask = impossible value, to be set to NaN."""
    rules = []
    for col in DAY_COLS:
        rules.append(("negative day interval", col, df[col] < 0))
    for col in COUNT_COLS:
        rules.append(("negative count", col, df[col] < 0))
    for col in AMOUNT_COLS:
        rules.append(("placeholder 1e9", col, df[col] >= PLACEHOLDER))
        rules.append(("negative amount", col, df[col] < 0))

    mean, total, count = df["prior_mean_purchase_value"], df["prior_total_spend"], df["prior_purchase_count"]
    expected_mean = total / count.where(count > 0)
    rules.append(("mean != total / count", "prior_mean_purchase_value",
                  expected_mean.notna() & ~np.isclose(mean, expected_mean, rtol=1e-4, atol=TOL)))
    rules.append(("max > total spend", "prior_max_purchase_value",
                  df["prior_max_purchase_value"] > total + TOL))
    rules.append(("365d spend > total spend", "spend_365d", df["spend_365d"] > total + TOL))
    rules.append(("365d spend < 180d spend", "spend_365d", df["spend_365d"] < df["spend_180d"] - TOL))
    return rules


def set_invalid_to_nan(df):
    """Apply every rule in order. The log counts only cells that were not already NaN (no double counting)."""
    df = df.copy()
    log = []
    for rule, col, mask in invalid_value_rules(df):
        hit = mask & df[col].notna()
        df.loc[hit, col] = np.nan
        log.append((rule, col, int(hit.sum())))
    return df, pd.DataFrame(log, columns=["rule", "column", "cells"])

In [10]:
train_3, invalid_log_train = set_invalid_to_nan(train_2)
test_3, invalid_log_test = set_invalid_to_nan(test_2)

side_by_side(invalid_log_train, invalid_log_test, keys=["rule", "column"])

,rule,column,cells train,cells test
0,365d spend > total spend,spend_365d,39,7
1,max > total spend,prior_max_purchase_value,45,6
2,mean != total / count,prior_mean_purchase_value,46,7
3,negative amount,prior_mean_purchase_value,16,2
4,negative amount,spend_365d,15,2
5,negative day interval,days_since_previous_purchase,16,2
6,negative day interval,prior_mean_gap_days,16,2
7,placeholder 1e9,prior_mean_purchase_value,16,2
8,placeholder 1e9,spend_365d,15,2


**Reading the table**
- Only 5 columns are affected. Every other column passes every rule.
- `prior_mean_purchase_value` and `spend_365d` are hit by several error types. They are the most corrupted columns.
- Test has the same patterns as train in smaller numbers. This confirms the rules must be part of `clean()` and applied to both sets.

# <font color='#E8800A'>4. Rebuild values from exact identities</font> <a class="anchor" id="rebuild"></a>
[Back to TOC](#toc)

**Idea:** several columns are linked by **exact** relationships. When one value is missing, or was just set to `NaN`, the others give it back **exactly**. This is deterministic, so it belongs in cleaning, not imputation.

| Identity | Holds when | Rebuilds |
|---|---|---|
| mean = total / count | count ≥ 1 | `prior_mean_purchase_value` |
| max = total | count = 1 | `prior_max_purchase_value` |
| days since previous = tenure | count = 1 | `days_since_previous_purchase` |
| tenure − days since previous = (count − 1) × mean gap | count ≥ 2 | `days_since_previous_purchase`, `prior_mean_gap_days` |
| spend 365d = spend 180d | same purchase count in both windows | `spend_365d` |
| spend 365d = total spend | every prior purchase is in the last 365 days | `spend_365d` |

*count = `prior_purchase_count`, tenure = `customer_tenure_days`*

**First, check** that each identity really holds on every row where all its values are present and valid:

In [11]:
def identity_checks(df):
    """For each identity: rows where it applies with all values present, and the share where it holds."""
    count, total, tenure = df["prior_purchase_count"], df["prior_total_spend"], df["customer_tenure_days"]
    days, gap = df["days_since_previous_purchase"], df["prior_mean_gap_days"]
    # name: (rows where the identity applies, left side, right side)
    identities = {
        "mean = total / count": (count >= 1, df["prior_mean_purchase_value"], total / count.where(count > 0)),
        "max = total (count = 1)": (count == 1, df["prior_max_purchase_value"], total),
        "days since = tenure (count = 1)": (count == 1, days, tenure),
        "tenure - days since = (count-1) x gap": (count >= 2, tenure - days, (count - 1) * gap),
        "spend 365d = spend 180d (same counts)": (df["purchase_count_365d"] == df["purchase_count_180d"],
                                                  df["spend_365d"], df["spend_180d"]),
        "spend 365d = total (all in 365d)": (df["purchase_count_365d"] == count, df["spend_365d"], total),
    }
    rows = {}
    for name, (applies, left, right) in identities.items():
        complete = applies & left.notna() & right.notna()
        holds = np.isclose(left[complete], right[complete], rtol=1e-4, atol=TOL)
        rows[name] = {"rows checked": int(complete.sum()), "share where it holds": holds.mean()}
    return pd.DataFrame(rows).T.astype({"rows checked": int})


# Checked after step 3, so the corrupted values are already NaN and do not count
identity_checks(pd.concat([train_3, test_3]))

,rows checked,share where it holds
mean = total / count,6517,1.0
max = total (count = 1),466,1.0
days since = tenure (count = 1),446,1.0
tenure - days since = (count-1) x gap,5908,1.0
spend 365d = spend 180d (same counts),1905,1.0
spend 365d = total (all in 365d),1760,1.0


**Every identity holds on 100% of the rows checked**, in train and test together. That makes them safe to use for rebuilding.

**Fix:** fill a `NaN` **only** where the identity applies and every value it needs is present. Everything else stays `NaN`.

In [12]:
def rebuild_from_identities(df):
    """Fill NaNs that an exact identity gives back. Returns the frame and a log of cells filled."""
    df = df.copy()
    log = []

    def fill(col, rule, values, applies):
        hit = df[col].isna() & applies & values.notna()
        df.loc[hit, col] = values[hit]
        log.append((rule, col, int(hit.sum())))

    count, total, tenure = df["prior_purchase_count"], df["prior_total_spend"], df["customer_tenure_days"]

    fill("prior_mean_purchase_value", "total / count", total / count.where(count > 0), count >= 1)
    fill("prior_max_purchase_value", "max = total (count = 1)", total, count == 1)
    fill("days_since_previous_purchase", "days since = tenure (count = 1)", tenure, count == 1)
    fill("days_since_previous_purchase", "tenure - (count-1) x gap",
         (tenure - (count - 1) * df["prior_mean_gap_days"]).round(), count >= 2)
    fill("prior_mean_gap_days", "(tenure - days since) / (count-1)",
         (tenure - df["days_since_previous_purchase"]) / (count - 1), count >= 2)
    fill("spend_365d", "spend 365d = spend 180d (same counts)", df["spend_180d"],
         df["purchase_count_365d"] == df["purchase_count_180d"])
    fill("spend_365d", "spend 365d = total (all in 365d)", total, df["purchase_count_365d"] == count)

    return df, pd.DataFrame(log, columns=["rule", "column", "cells"])

In [13]:
train_4, rebuild_log_train = rebuild_from_identities(train_3)
test_4, rebuild_log_test = rebuild_from_identities(test_3)

side_by_side(rebuild_log_train, rebuild_log_test, keys=["rule", "column"])

,rule,column,cells train,cells test
0,(tenure - days since) / (count-1),prior_mean_gap_days,203,32
1,days since = tenure (count = 1),days_since_previous_purchase,22,1
2,max = total (count = 1),prior_max_purchase_value,3,0
3,spend 365d = spend 180d (same counts),spend_365d,80,4
4,spend 365d = total (all in 365d),spend_365d,28,1
5,tenure - (count-1) x gap,days_since_previous_purchase,179,31
6,total / count,prior_mean_purchase_value,272,41


**Reading the table**
- The rebuild fills both the values set to `NaN` in step 3 **and** some of the values that were already missing in the raw data. Those are exact values, so nothing has to be imputed for them later.
- `prior_mean_purchase_value` is fully recovered wherever the customer has at least one prior purchase.
- **First purchases** (`prior_purchase_count == 0`) keep `NaN` in the gap and value columns on purpose. There is no previous purchase, so that `NaN` is information. It is handled in `04_preprocessing` (missing indicator / `is_first_purchase` flag).

In [14]:
# Rebuilt values must themselves be valid: rerun the rules of step 3, nothing should be flagged
_, recheck = set_invalid_to_nan(pd.concat([train_4, test_4]))
assert recheck["cells"].sum() == 0, recheck[recheck["cells"] > 0]
print("rebuilt values pass every rule of step 3")

rebuilt values pass every rule of step 3


# <font color='#E8800A'>5. Categorical labels</font> <a class="anchor" id="categories"></a>
[Back to TOC](#toc)

**Question:** are there labels that differ only in case, spacing or punctuation (`"Zone 4"` vs `"zone 4 "`) and should be merged?

**Check:** count the distinct labels before and after normalising (lower case, single spaces, no punctuation), on train and test together.

In [15]:
both = pd.concat([train_4, test_4])
normalised = lambda s: s.str.lower().str.replace(r"[^a-z0-9]+", " ", regex=True).str.strip()

pd.DataFrame({
    "distinct labels": both[CATEGORICAL].nunique(),
    "after normalising": both[CATEGORICAL].apply(normalised).nunique(),
    "missing (train)": train_4[CATEGORICAL].isna().sum(),
    "missing (test)": test_4[CATEGORICAL].isna().sum(),
    "levels only in test": [len(set(test_4[c].dropna()) - set(train_4[c].dropna())) for c in CATEGORICAL],
})

,distinct labels,after normalising,missing (train),missing (test),levels only in test
payment_type,12,12,195,30,3
salesperson,9,9,194,30,0
commercial_zone,7,7,0,0,2
transport_method,82,82,196,30,2
warehouse,3,3,195,30,1
document_series,3,3,0,0,0


**Result:** normalising does not merge any labels, so **no fix is needed**. The labels are already consistent.

Left for `04_preprocessing` (they learn from the data or depend on the model):
- the ~3% missing values → imputation inside the Pipeline;
- rare levels (e.g. 82 `transport_method` levels) → grouping;
- levels only in test → `handle_unknown="ignore"` / encoder fallback.

# <font color='#E8800A'>6. The full `clean()` function and checks</font> <a class="anchor" id="clean"></a>
[Back to TOC](#toc)

**All steps chained in one function**, the version to move to `src/cleaning.py`:
- `clean(df)` → rename and types → impossible values to `NaN` → rebuild. Same for train and test, never drops rows.
- `drop_empty_rows(df)` → train only, called **before** `clean`.

In [16]:
def clean(df):
    """Deterministic cleaning, identical for train and test. Never drops rows."""
    df = rename_and_type(df)
    df, _ = set_invalid_to_nan(df)
    df, _ = rebuild_from_identities(df)
    return df


train_clean = clean(drop_empty_rows(train_raw)[0])
test_clean = clean(test_raw)

print(f"train: {train_raw.shape} -> {train_clean.shape}")
print(f"test:  {test_raw.shape} -> {test_clean.shape}")

train: (6534, 38) -> (6469, 38)
test:  (986, 37) -> (986, 37)


**Sanity checks** (an `AssertionError` means something broke):
1. `clean()` gives the same result as the step-by-step version above.
2. Test keeps every row.
3. Train and test have the same columns (apart from the target).
4. Running `clean()` twice changes nothing (idempotent).
5. No impossible value is left.
6. The target is untouched.

In [17]:
assert train_clean.equals(train_4) and test_clean.equals(test_4)                  # 1
assert len(test_clean) == len(test_raw)                                           # 2
assert list(train_clean.columns.drop(TARGET)) == list(test_clean.columns)         # 3
assert clean(train_clean).equals(train_clean) and clean(test_clean).equals(test_clean)  # 4
assert set_invalid_to_nan(pd.concat([train_clean, test_clean]))[1]["cells"].sum() == 0  # 5
assert train_clean[TARGET].equals(train_raw.loc[train_clean.index, TARGET])       # 6
print("all checks passed")

all checks passed


# <font color='#E8800A'>7. Summary</font> <a class="anchor" id="summary"></a>
[Back to TOC](#toc)

### Rules applied

In [18]:
summary = pd.concat([
    pd.DataFrame([{"step": "2. drop empty rows", "rule": "every feature missing", "column": "(whole row)",
                   "cells train": n_empty, "cells test": 0}]),
    side_by_side(invalid_log_train, invalid_log_test, ["rule", "column"]).assign(step="3. impossible -> NaN"),
    side_by_side(rebuild_log_train, rebuild_log_test, ["rule", "column"]).assign(step="4. rebuilt exactly"),
], ignore_index=True)
summary[["step", "rule", "column", "cells train", "cells test"]]

,step,rule,column,cells train,cells test
0,2. drop empty rows,every feature missing,(whole row),65,0
1,3. impossible -> NaN,365d spend > total spend,spend_365d,39,7
2,3. impossible -> NaN,max > total spend,prior_max_purchase_value,45,6
3,3. impossible -> NaN,mean != total / count,prior_mean_purchase_value,46,7
4,3. impossible -> NaN,negative amount,prior_mean_purchase_value,16,2
5,3. impossible -> NaN,negative amount,spend_365d,15,2
6,3. impossible -> NaN,negative day interval,days_since_previous_purchase,16,2
7,3. impossible -> NaN,negative day interval,prior_mean_gap_days,16,2
8,3. impossible -> NaN,placeholder 1e9,prior_mean_purchase_value,16,2
9,3. impossible -> NaN,placeholder 1e9,spend_365d,15,2


### Missing values before and after cleaning

This is what `04_preprocessing` still has to deal with. *(train before = after dropping the 65 empty rows, so the comparison is fair)*

In [19]:
first_train = train_clean["prior_purchase_count"] == 0
first_test = test_clean["prior_purchase_count"] == 0

missing = pd.DataFrame({
    "train before": train_2.isna().sum(),
    "train after": train_clean.isna().sum(),
    "test before": test_2.isna().sum(),
    "test after": test_clean.isna().sum(),
    # how many of the remaining train NaNs are first purchases (structural, carry information)
    "train after, first purchases": train_clean[first_train].isna().sum(),
})
missing = missing.fillna(0).astype(int)  # the target column exists only in train
missing = missing[missing.drop(columns="train after, first purchases").sum(axis=1) > 0]
missing.sort_values("train after", ascending=False)

,train before,train after,test before,test after,"train after, first purchases"
prior_std_gap_days,1413,1413,69,69,603
prior_mean_gap_days,1243,1056,76,46,603
prior_std_purchase_value,1048,1048,46,46,603
prior_max_purchase_value,603,645,22,28,603
days_since_previous_purchase,796,611,52,22,603
prior_mean_purchase_value,797,603,52,22,603
prior_min_purchase_value,603,603,22,22,603
transport_method,196,196,30,30,23
payment_type,195,195,30,30,19
warehouse,195,195,30,30,13


**Reading the table**
- `prior_mean_purchase_value`, `prior_min/max_purchase_value` and most gap/recency columns are now missing **only for first purchases** (603 in train). That missingness is structural.
- `days_since_previous_purchase`: 796 → 611 missing in train, i.e. the 603 first purchases plus 8 rows that could not be rebuilt.
- The `prior_std_*` columns did not change: no rule or identity touches them.
- Categoricals keep their ~3% scattered `NaN`.

### What is left for later

| Remaining issue | Where | How |
|---|---|---|
| Structural `NaN` for first purchases (gap, value, std columns) | `04_preprocessing` | `is_first_purchase` flag / missing indicator, then fill |
| `prior_std_*` undefined with 1 or 2 prior purchases | `04_preprocessing` | same as above (structural) |
| Scattered ~3% `NaN` (categoricals, part of `spend_365d`, history columns) | `04_preprocessing` | imputation **inside** the Pipeline (fitted on the training fold only) |
| Zero cost (486 rows), cost > net (39 rows) | `05_features` | plausible (free samples, sales at a loss) → kept, flag as features |
| Strong skew / zero-inflated amounts | `04_preprocessing` | `log1p` / robust scaling |
| `random_noise`, `ID` | `05_features` | drop (EDA evidence) |

### Lines for `docs/decisions.md`

| Decision | Alternatives considered | Evidence |
|---|---|---|
| Rename Portuguese columns to English snake_case | keep original names | readability; mapping in `cleaning_ludovico` §1 |
| Drop the 65 empty train rows | keep and impute | no features, cause all 10 duplicates; test has none (§2) |
| Impossible values → `NaN` via general domain rules | clip, abs(), hard-code −4 / 1e9 | −4 days, 1e9, sign flips, mean ≠ total/count, max > total, 365d > total (§3) |
| Rebuild values from exact identities before imputing | impute everything in the Pipeline | identities hold on 100% of complete rows (§4) |
| No changes to categorical labels | manual mapping | normalising merges no labels (§5) |

### Next step
Move `COLUMN_MAP`, the column lists and the five functions (`rename_and_type`, `drop_empty_rows`, `set_invalid_to_nan`, `rebuild_from_identities`, `clean`) to `src/cleaning.py`. Then later notebooks only need:
```python
from src.cleaning import clean, drop_empty_rows
train = clean(drop_empty_rows(train_raw)[0])
test = clean(test_raw)
```